In [ ]:
# In this Jupiter notebook, we will fine-tune the model on the Magicoder dataset.

# Magicoder dataset is a dataset of instruction, input, and output triples. 
# It is a dataset of 1,000 examples. We will use this dataset to fine-tune the model on the Magicoder dataset.
# It is trainned with multiple programming languages like Python, Java, C++, etc.
# I will fine tune this data set only to generate Python code and restric the model to generate only Python code.
# Model will throw proper warnings if user asks for other programming languages.

# Here are list of steps involved in fine-tuning the model with any dataset:
# 1. Identify the data set that you want to use in fine tuning the model.
# 2. Understand the data set and its format.Which is curcial for fine-tuning the model, beacuse you need to convert 
#   the data set to the format that the model expects.
# 3. Pick the model you want to fine-tune and tokenizer of the model.
# 4. Understand the model prompting template format - you need to convert the data set to the format that the model expects.
# 5. Write a function to convert the data set to the format that the model expects.
# 6. Load the data set and convert it to the format that the model expects.
# 7. Identify the parameters/properties of the model that you want to fine-tune. Most important parameters are:
# 7.1 QLoRA or LoRA configuration
# 7.2 Model specific configuration like mode name, tokenizer name and fine tunned model name etc
# 7.3 Quantization configuration like quantization type and quantization bits etc
# 7.4 Training configuration like number of epochs, batch size, learning rate etc
# 7.5 SFT parameters like prompt template, model details, QLoRA parameters, bits and bytes, trainable parameters etc
# 8. Train the model on the data set.
# 9. Evaluate the model on the data set.
# 10. Save the model and tokenizer to disk.
# 11. Merge the base model and tokenizer with the fine-tuned model and tokenizer.
# 12. Clean up the memory and unload the model and tokenizer.
# 13. Reload the merged model and tokenizer from disk.
# 14. Evaluate the fine tunned model with user prompts.

In [ ]:
# try testing hugging face community connectivity using python code

from huggingface_hub import notebook_login

notebook_login()

# List the models available in the Hugging Face Hub starting with "llama" using python code

from huggingface_hub import list_models

hf_models = list_models()

# print the models available in the Hugging Face Hub from HfApi.list_models

# HF_TOKEN from environment variable
import os
hf_token = os.environ["HF_TOKEN"]
print(f"Token from environment: {hf_token[:20]}...")

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from trl import SFTTrainer


In [ ]:

dataset_name = "ise-uiuc/Magicoder-OSS-Instruct-75K"
dataset = load_dataset(dataset_name, split="train")


In [ ]:
# Look at the data set
print(dataset)

# Look at the first 3 examples
for i in range(3):
    print(dataset[i])

# Look at the last 3 examples
for i in range(len(dataset) - 3, len(dataset)):
    print(dataset[i])

In [ ]:
# restrict dataset only for python code generation
dataset = dataset.filter(lambda example: example["lang"] == "python")
print(dataset)

# Look at the first 3 examples
for i in range(3):
    print(dataset[i])

# print probem and solution
for i in range(3):
    print(f"Problem: {dataset[i]['problem']}\nSolution: {dataset[i]['solution']}\n") # print(dataset[i]["problem"])


In [ ]:
# define model configuration and tokenizer parameters
from os import device_encoding

from click import prompt


model_config = {
    "model_name_or_path": "meta-llama/Llama-2-7b-chat-hf",
    "tokenizer_name_or_path": "meta-llama/Llama-2-7b-chat-hf",
    "use_auth_token": True,
    "trust_remote_code": True,
    "device_map": "auto",
    "new_model_name": "llama-2-7b-finetuned-magicoder",
    "merged_model_name": "llama-2-7b-finetuned-magicoder-merged",
    "data_set_name": "ise-uiuc/Magicoder-OSS-Instruct-75K",
}

# define QLoRA parameters
QLoRA_paramters = {
    "lora_r": 64,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "lora_target_modules": ["q_proj", "v_proj"],
    "bias": "none",
    "task_type": "CAUSAL_LM",
}

# quantization parameters
bits_and_bytes = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "use_nested_quant": False,
    "bnb_4bit_compute_dtype": torch.float16,
    "use_4bit": True,
}

# training parameters
training_parameters = {
    "num_train_epochs": 3,
    "output_dir": "./results",
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "save_steps": 100,
    "logging_steps": 10,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "weight_decay": 0.1,
    "max_grad_norm": 0.3,
    "fp16": True,
    "bf16": False,
    "bf16_full_eval": False,
    "optim": "paged_adamw_32bit",
    "logging_dir": "logs",
    "report_to": "tensorboard",
    "max_steps": 10
}

# def SFT fine-tuning parameters
fine_tuning_parameters = {
    "model_details": model_config,
    "QLoRA_parameters": QLoRA_paramters,
    "bits_and_bytes": bits_and_bytes,
    "training_parameters": training_parameters,
    "max_seq_length": None,
    "packing" : False,
    "device_map": "auto"
}

In [ ]:
# Prompt template for model fine-tuning: "meta-llama/Llama-2-7b-hf" using python code
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_config["model_name_or_path"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

messages = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What is the capital of Italy?"},
    {"role": "assistant", "content": "The capital of Italy is Rome."},
    {"role": "user", "content": "What is the capital of Germany?"},
    {"role": "assistant", "content": "The capital of Germany is Berlin."},
    {"role": "user", "content": "What is the capital of Spain?"},
    {"role": "assistant", "content": "The capital of Spain is Madrid."},
    {"role": "user", "content": "What is the capital of Portugal?"}
]

formatted_prompt = tokenizer.apply_chat_template(messages, tokenizer=False, add_generation_prompt=True)
print(formatted_prompt)
print(tokenizer.decode(formatted_prompt["input_ids"]))


In [ ]:
#model_config["prompt_template"] = """<s><INST> <<sys>>{system_message}<</sys>> 
# <<user>>{user_message}<</user>> [/INST] {model_answer} </s>"""

In [ ]:
# convert the dataset to the format that the model expects
# data set is having problem and solution and we need to convert it to the format that the model expects

# 1. convert the problem and solution to the format that the model expects
# 2. convert the problem and solution to the format that the model expects

def convert_to_prompt_format(example):
    # Enforce strict Python parameters inside the Llama-2 System Frame
    system_msg = (
        "You are a Python automation assistant. You only support and generate Python code. "
        "If a user asks you for code in any other language (like Java, C++, or JavaScript), "
        "you must explicitly refuse and remind them of your Python specialization boundaries."
    )
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": example["problem"]},
        {"role": "assistant", "content": example["solution"]}
    ]
    
    # Generate structural text using Llama-2 standard chat syntax templates
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": formatted_text}

# print datset columns
print(dataset.column_names)
dataset = dataset.map(convert_to_prompt_format, remove_columns=dataset.column_names)
print(dataset)

In [ ]:
# Load the model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_config["model_name_or_path"])
tokenizer = AutoTokenizer.from_pretrained(model_config["model_name_or_path"])

In [ ]:
# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=bits_and_bytes["load_in_4bit"],
    bnb_4bit_quant_type=bits_and_bytes["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=bits_and_bytes["bnb_4bit_compute_dtype"],
    bnb_4bit_use_double_quant=bits_and_bytes["bnb_4bit_use_double_quant"]
)

from peft import LoraConfig
import peft
# LoRA configuration
peft_config = LoraConfig(
    r=fine_tuning_parameters["QLoRA_parameters"]["lora_r"],
    lora_alpha=fine_tuning_parameters["QLoRA_parameters"]["lora_alpha"],
    target_modules=fine_tuning_parameters["QLoRA_parameters"]["lora_target_modules"],
    lora_dropout=fine_tuning_parameters["QLoRA_parameters"]["lora_dropout"],
    bias=fine_tuning_parameters["QLoRA_parameters"]["bias"],
    task_type=fine_tuning_parameters["QLoRA_parameters"]["task_type"],
)

In [ ]:
# Load the model and tokenizer
from torch import mode


model = AutoModelForCausalLM.from_pretrained(model_config["model_name_or_path"], 
                                             quantization_config=quantization_config,
                                             device_map=model_config["device_map"])

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_config["model_name_or_path"])

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = model.prepare_model_for_kbit_training(model)

In [ ]:
# train the model on the data set

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=model_config["model_name_or_path"],
    per_device_train_batch_size=training_parameters["per_device_train_batch_size"],
    per_device_eval_batch_size=training_parameters["per_device_eval_batch_size"],
    learning_rate=training_parameters["learning_rate"],
    num_train_epochs=training_parameters["num_train_epochs"],
    warmup_ratio=training_parameters["warmup_ratio"],
    lr_scheduler_type=training_parameters["lr_scheduler_type"],
    weight_decay=training_parameters["weight_decay"],
    max_grad_norm=training_parameters["max_grad_norm"],
    fp16=training_parameters["fp16"],
    bf16=training_parameters["bf16"],
    bf16_full_eval=training_parameters["bf16_full_eval"],
    optim=training_parameters["optim"],
    logging_dir=training_parameters["logging_dir"],
    report_to=training_parameters["report_to"],
    # trust_remote_code=model_config["trust_remote_code"],
    # use_auth_token=model_config["use_auth_token"],
    # device_map=model_config["device_map"],
    # torch_dtype=model_config["torch_dtype"],
    # load_in_8bit=model_config["load_in_8bit"],
    max_steps=training_parameters["max_steps"],
    gradient_accumulation_steps=True
)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)
trainer.train()

In [ ]:
# save the model and tokenizer to disk
trainer.model.save_pretrained(model_config["new_model_name"])
tokenizer.save_pretrained(model_config["new_model_name"])

In [ ]:
# Clear the memory
import gc
gc.collect()
del model
del tokenizer
del trainer
torch.cuda.empty_cache()

In [ ]:
# merge the base model and tokenizer with the fine-tuned model and tokenizer

from peft import PeftModel
base_model_reload = AutoModelForCausalLM.from_pretrained(
    model_config["model_name_or_path"],
    torch_dtype=torch.float16,
    device_map="auto"
)

# Wrap base model using your newly saved adapters
peft_model = PeftModel.from_pretrained(base_model_reload, model_config["new_model_name"])

# Mathematically fuse weights together
merged_model = peft_model.merge_and_unload()

# Export clean unified production directory
merged_model.save_pretrained(model_config["merged_model_name"])
tokenizer_reload = AutoTokenizer.from_pretrained(model_config["model_name_or_path"])
tokenizer_reload.save_pretrained(model_config["merged_model_name"])

print("Standalone production model successfully compiled.")

In [ ]:
# validate the fine tunned model with user prompts
test_prompt = "<s>[INST] Can you write a fast string sorting function in Java? [/INST]"

inputs = tokenizer_reload(test_prompt, return_tensors="pt").to("cuda")
generate_ids = merged_model.generate(
    inputs.input_ids, 
    max_new_tokens=150,
    temperature=0.3,
    do_sample=True
)

generated_text = tokenizer_reload.batch_decode(generate_ids, skip_special_tokens=True)[0]
print(generated_text)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Load the fine-tuned and merged model
model_name = model_config["merged_model_name"]
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype=torch.float16,  # Or torch.bfloat16 if supported by your hardware
    device_map="auto"           # Automatically maps the model to your GPU (cuda:0)
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Define and format your prompt 
# NOTE: If you fine-tuned using ChatML or Llama markers, wrap your prompt in those tags here!
prompt = "Explain me the fine tunning LLM model with QLoRA and LoRA"

# 3. Tokenize and explicitly send tensors to the GPU (.to("cuda"))
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Generate response safely
# We switch from 'max_length=30' to 'max_new_tokens=256' to ensure a complete output
generate_ids = model.generate(
    inputs.input_ids, 
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True
)

# 5. Decode and print the output
generated_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True)[0]
print(generated_text)